In [2]:
from notepad import TraceDataGeneration

In [9]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten
import tensorflow.keras.backend as K


class CustomModel(Model):
    def __init__(self):
        super(CustomModel, self).__init__()
        # Define your layers and parameters here
        self.dense = Dense(64, activation='relu')
        self.output_layer = Dense(len(FUNCTION_LIST) * 2 + 1)  # 예상되는 파라미터 수에 따라 조절

    def call(self, inputs):
        x = self.dense(inputs)
        x = self.output_layer(x)
        return x

    def mse_loss(self, y_true, y_pred):
        # y_true는 원본 파라미터들입니다.
        # y_pred는 모델이 예측한 파라미터들입니다.
        
        # 원본 파라미터들을 TraceDataGeneration을 사용하여 생성된 함수로 변환
        generator = TraceDataGeneration(n=100, global_range=100, seed=1, jitter_bool=False)
        input_signals = generator.automated_generation_random_para(function_list)
        generated_params = []
        for func, param in zip(function_list, y_true):
            param_dict = dict(zip(['start_value', 'end_value', 't1', 't2', 'b', 'c', 'length'], param))
            generated_params.append(getattr(generator, func)(**param_dict))
        
        # 생성된 함수로부터 생성된 신호들을 조합
        combined_signals = np.vstack(generated_params)
        
        # MSE 계산
        mse = K.mean(K.square(combined_signals - input_signals))
        
        # R2 계산
        mean_true = K.mean(y_true)
        r_squared = 1 - K.sum(K.square(y_true - y_pred)) / K.sum(K.square(y_true - mean_true))
        
        # 조합된 손실 함수 (MSE와 R2의 가중 평균)
        combined_loss = 0.8 * mse + 0.2 * r_squared
        
        return combined_loss

# 생성된 TraceDataGeneration 객체를 사용하여 입력 신호 생성
generator = TraceDataGeneration(n=100, global_range=100, seed=1, jitter_bool=False)
function_list = generator.sample_function_list(10)
input_signals = generator.automated_generation_random_para(function_list)

# 각 함수에서 사용된 파라미터 추출
params = []
for func in function_list:
    params.append(generator.get_param_normal(func, mean_val=50))  # mean_val은 예시입니다. 실제로 사용할 값을 지정하세요.

# 파라미터를 1차원 배열로 변환
params_flat = [val for param in params for val in param.values()]

# CustomModel 생성
model = CustomModel()

# 모델 컴파일 (손실 함수를 mse_loss로 지정)
model.compile(optimizer='adam', loss=model.mse_loss)

# 모델 학습
model.fit(np.vstack(input_signals), np.array(params_flat), epochs=10, validation_split=0.2)

# 학습된 모델을 사용하여 예측
predicted_params = model.predict(np.vstack(input_signals))

# 각 함수에서 사용된 파라미터로 분리
num_params_per_function = 2  # 각 함수당 사용된 파라미터 수 (예시로 2로 설정)
predicted_params = np.split(predicted_params, len(function_list))
predicted_params = [dict(zip(params[0].keys(), param)) for param in predicted_params]

{'start_value': 55.14307743283224, 'high_value': 73.37440621348023, 'end_value': 62.098249829251834, 't1': 19, 't2': 54, 'b': 5, 'c': 7, 'length': 84}
{'start_value': 93.62978846316575, 'end_value': 100.0, 't1': 5, 'b': 3, 'length': 25}
{'start_value': 5.0, 'end_value': 10.0, 't1': 43, 't2': 61, 'length': 66}
{'start_value': 37.42031320783627, 'end_value': 47.76625334734285, 't1': 48, 'b': 7, 'length': 59}
{'start_value': 19.43566662820613, 'high_value': 37.81482513397573, 'end_value': 28.324841735324707, 't1': 16, 't2': 33, 't3': 60, 't4': 63, 'length': 74}
{'start_value': 17.968866400649578, 'high_value': 26.321328600748522, 'end_value': 21.524664234764334, 't1': 6, 't2': 31, 't3': 55, 't4': 61, 'length': 67}
{'start_value': 25.70248445366691, 'high_value': 51.75711011288421, 'end_value': 33.26144115056057, 't1': 10, 't2': 15, 't3': 23, 't4': 25, 'length': 31}
{'start_value': 48.14311393331147, 'end_value': 53.1681961920746, 't1': 67, 'length': 74}
{'start_value': 54.56717216826025, 

ValueError: Failed to convert a NumPy array to a Tensor (Unsupported object type int).

In [12]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input
import tensorflow.keras.backend as K

class CustomModel(Model):
    def __init__(self):
        super(CustomModel, self).__init__()
        # Define your layers and parameters here
        self.dense = Dense(64, activation='relu')
        self.output_layer = Dense(len(FUNCTION_LIST) * 2 + 1)  # 예상되는 파라미터 수에 따라 조절

    def call(self, inputs):
        x = self.dense(inputs)
        x = self.output_layer(x)
        return x

    def mse_loss(self, y_true, y_pred):
        # y_true는 원본 파라미터들입니다.
        # y_pred는 모델이 예측한 파라미터들입니다.
        
        # 원본 파라미터들을 TraceDataGeneration을 사용하여 생성된 함수로 변환
        generator = TraceDataGeneration(n=100, global_range=100, seed=1, jitter_bool=False)
        input_signals = generator.automated_generation_random_para(function_list)
        generated_params = []
        for func, param in zip(function_list, y_true):
            param_dict = dict(zip(['start_value', 'end_value', 't1', 't2', 'b', 'c', 'length'], param))
            param_dict['start_value'] = float(param_dict['start_value'])
            param_dict['end_value'] = float(param_dict['end_value'])
            param_dict['t1'] = int(param_dict['t1'])
            param_dict['t2'] = int(param_dict['t2'])
            param_dict['b'] = int(param_dict['b'])
            param_dict['c'] = int(param_dict['c'])
            param_dict['length'] = int(param_dict['length'])
            generated_params.append(getattr(generator, func)(**param_dict))
        
        # 생성된 함수로부터 생성된 신호들을 조합
        combined_signals = np.vstack(generated_params)
        
        # MSE 계산
        mse = K.mean(K.square(combined_signals - input_signals))
        
        # R2 계산
        mean_true = K.mean(y_true)
        r_squared = 1 - K.sum(K.square(y_true - y_pred)) / K.sum(K.square(y_true - mean_true))
        
        # 조합된 손실 함수 (MSE와 R2의 가중 평균)
        combined_loss = 0.8 * mse + 0.2 * r_squared
        
        return combined_loss

# CustomModel 클래스 정의

# 생성된 TraceDataGeneration 객체를 사용하여 입력 신호 생성
generator = TraceDataGeneration(n=100, global_range=100, seed=1, jitter_bool=False)
function_list = generator.sample_function_list(10)
input_signals = generator.automated_generation_random_para(function_list)
input_signals = [df['PARAMETER_VALUE'].values for df in input_signals]

# 각 함수에서 사용된 파라미터 추출
params = []
for func in function_list:
    params.append(generator.get_param_normal(func, mean_val=50))

# 파라미터를 1차원 배열로 변환
params_flat = [val for param in params for val in param.values()]

# CustomModel 생성
model = CustomModel()

# 모델 컴파일 (손실 함수를 mse_loss로 지정)
model.compile(optimizer='adam', loss=model.mse_loss)

# 모델 학습
input_signals = np.array(input_signals)  # 이 부분을 추가
params_flat = np.array(params_flat)      # 이 부분을 추가
model.fit(input_signals, params_flat, epochs=10, validation_split=0.2)

# 학습된 모델을 사용하여 예측
predicted_params = model.predict(input_signals)

# 각 함수에서 사용된 파라미터로 분리
num_params_per_function = 2  # 각 함수당 사용된 파라미터 수 (예시로 2로 설정)
predicted_params = np.split(predicted_params, len(function_list))
predicted_params = [dict(zip(params[0].keys(), param)) for param in predicted_params]

{'start_value': 55.14307743283224, 'high_value': 73.37440621348023, 'end_value': 62.098249829251834, 't1': 19, 't2': 54, 'b': 5, 'c': 7, 'length': 84}
{'start_value': 93.62978846316575, 'end_value': 100.0, 't1': 5, 'b': 3, 'length': 25}
{'start_value': 5.0, 'end_value': 10.0, 't1': 43, 't2': 61, 'length': 66}
{'start_value': 37.42031320783627, 'end_value': 47.76625334734285, 't1': 48, 'b': 7, 'length': 59}
{'start_value': 19.43566662820613, 'high_value': 37.81482513397573, 'end_value': 28.324841735324707, 't1': 16, 't2': 33, 't3': 60, 't4': 63, 'length': 74}
{'start_value': 17.968866400649578, 'high_value': 26.321328600748522, 'end_value': 21.524664234764334, 't1': 6, 't2': 31, 't3': 55, 't4': 61, 'length': 67}
{'start_value': 25.70248445366691, 'high_value': 51.75711011288421, 'end_value': 33.26144115056057, 't1': 10, 't2': 15, 't3': 23, 't4': 25, 'length': 31}
{'start_value': 48.14311393331147, 'end_value': 53.1681961920746, 't1': 67, 'length': 74}
{'start_value': 54.56717216826025, 

C:\Users\HYKP\AppData\Local\Temp/ipykernel_17752/4148293029.py:75: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  input_signals = np.array(input_signals)  # 이 부분을 추가


ValueError: Failed to convert a NumPy array to a Tensor (Unsupported object type numpy.ndarray).

In [3]:
# Trace 생성
generator = TraceDataGeneration(n=100, global_range=100)
num_traces = 10  # 생성할 trace 수

traces = []
for _ in range(num_traces):
    trace = generator.automated_generation_random_para(step_num=5, jitter=True)
    traces.append(trace)

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# 간단한 딥러닝 모델 생성
model = models.Sequential([
    layers.Flatten(input_shape=(len(traces[0]),)),  # 입력 형태에 맞게 조정
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(30, activation='softmax')  # 30은 예측할 클래스 수, 예시로 임의로 선택
])

# 모델 컴파일
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# trace 데이터를 모델에 입력으로 넣을 준비
X_train = np.array(traces)
y_train = np.array([0, 1, 2, 1, 0, 2, 1, 0, 2, 1])  # 각 trace에 대한 정답 레이블, 예시로 임의로 선택

# 모델 훈련
model.fit(X_train, y_train, epochs=10)

C:\Users\HYKP\AppData\Local\Temp/ipykernel_12576/3016611442.py:19: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  X_train = np.array(traces)


ValueError: Failed to convert a NumPy array to a Tensor (Unsupported object type numpy.ndarray).

In [ ]:
# 테스트할 trace 데이터
test_trace = generator.automated_generation_random_para(step_num=5, jitter=True)

# 모델을 사용하여 예측
predicted_class = model.predict(np.array([test_trace]))